## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


def get_device():
    """
      Return the computing device used in the current session.
    """
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))


CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Using device: cuda
Tesla T4


## 4. Load Tokenizer

In [4]:
# %%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""

''' Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()'''
import re
from tqdm import tqdm
from collections import defaultdict, Counter

class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = vocab_size
        self.reserved = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos = list(self.reserved)
        self.stoi = {}
        self.ranks = {}
        self.merged_map = {}
        self._cache = {}
        # Regex pattern matching words, numbers, individual punctuation, and whitespace sequences
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        self.pat = re.compile(pattern, re.UNICODE)
        
    def _split_text(self, text):
        """Splits text into chunks preserving whitespace, or falls back to basic regex split."""
        pattern = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\s\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
        return re.findall(pattern, str(text), flags=re.UNICODE)

    def fit(self, texts, max_unique_words=200000):
        """Train a byte-level BPE tokenizer using inverted indexing for fast merging."""
        raw_counts = Counter()
        iterator = [texts] if isinstance(texts, str) else texts

        # Preserve whitespace tokens when populating initial raw_counts
        for text in tqdm(iterator, desc="Building word vocabulary"):
            for chunk in self._split_text(text):
                raw_counts[chunk] += 1

        if max_unique_words and len(raw_counts) > max_unique_words:
            most_common = raw_counts.most_common(max_unique_words)
            raw_counts = dict(most_common)

        word_freqs = {}
        pair_counts = defaultdict(int)
        pair_to_words = defaultdict(set)

        for word, count in raw_counts.items():
            encoded = word.encode("utf-8")
            byte_word = tuple(encoded[i:i+1] for i in range(len(encoded)))
            word_freqs[byte_word] = count

            for i in range(len(byte_word) - 1):
                pair = (byte_word[i], byte_word[i + 1])
                pair_counts[pair] += count
                pair_to_words[pair].add(byte_word)

        base_bytes = sorted({token for word in word_freqs for token in word})
        self.itos = list(self.reserved) + base_bytes
        self.merges = []

        target_merges = self.vocab_size - len(self.itos)

        with tqdm(total=target_merges, desc="Training BPE Merges") as pbar:
            while len(self.itos) < self.vocab_size:
                if not pair_counts:
                    break

                best_pair = max(pair_counts, key=lambda p: (pair_counts[p], p[0], p[1]))
                if pair_counts[best_pair] <= 0:
                    break

                merged_token = best_pair[0] + best_pair[1]
                self.merges.append((best_pair, merged_token))
                self.itos.append(merged_token)

                words_to_update = list(pair_to_words[best_pair])
                del pair_counts[best_pair]
                del pair_to_words[best_pair]

                p0, p1 = best_pair

                for word in words_to_update:
                    count = word_freqs.get(word, 0)
                    if count == 0:
                        continue

                    new_word = []
                    i = 0
                    n = len(word)

                    while i < n:
                        if i + 1 < n and word[i] == p0 and word[i + 1] == p1:
                            new_word.append(merged_token)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1

                    new_word = tuple(new_word)

                    for j in range(len(word) - 1):
                        old_p = (word[j], word[j + 1])
                        if old_p in pair_counts:
                            pair_counts[old_p] -= count
                            if pair_counts[old_p] <= 0:
                                del pair_counts[old_p]

                    for j in range(len(new_word) - 1):
                        new_p = (new_word[j], new_word[j + 1])
                        pair_counts[new_p] += count
                        pair_to_words[new_p].add(new_word)

                    del word_freqs[word]
                    word_freqs[new_word] = count

                pbar.update(1)

        self.vocab_size = len(self.itos)
        self.stoi = {token: idx for idx, token in enumerate(self.itos)}
        self.ranks = {pair: rank for rank, (pair, _) in enumerate(self.merges)}
        self.merged_map = {pair: merged for pair, merged in self.merges}
        self._cache = {}
        return self

    def encode_word(self, word):
        """Encode single word using memoization and local pair updates."""
        if word in self._cache:
            return self._cache[word]

        encoded = word.encode("utf-8")
        tokens = [encoded[i:i+1] for i in range(len(encoded))]

        if len(tokens) <= 1:
            res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
            self._cache[word] = res
            return res

        while len(tokens) > 1:
            min_rank = float("inf")
            min_pair = None

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])
                rank = self.ranks.get(pair, float("inf"))
                if rank < min_rank:
                    min_rank = rank
                    min_pair = pair

            if min_pair is None or min_rank == float("inf"):
                break

            merged = self.merged_map[min_pair]
            new_tokens = []
            i = 0
            n = len(tokens)
            p0, p1 = min_pair

            while i < n:
                if i + 1 < n and tokens[i] == p0 and tokens[i + 1] == p1:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        res = [self.stoi.get(token, self.stoi["<unk>"]) for token in tokens]
        self._cache[word] = res
        return res

    def encode(self, text):
        """Encode a sentence while preserving original spaces and newlines."""
        ids = [self.stoi["<s>"]]
        for chunk in self._split_text(text):
            ids.extend(self.encode_word(chunk))
        ids.append(self.stoi["</s>"])
        return ids

    def decode(self, ids):
        """Decode BPE token IDs back into text seamlessly."""
        byte_list = []
        for idx in ids:
            if 0 <= idx < len(self.itos):
                tok = self.itos[idx]
            else:
                tok = "<unk>"
            if tok not in self.reserved and isinstance(tok, bytes):
                byte_list.append(tok)
        try:
            return b"".join(byte_list).decode("utf-8", errors="ignore")
        except Exception:
            return ""

    def get_vocab_size(self):
        return self.vocab_size

def train_tokenizer(train_text: str, vocab_size: int):
    """Train A-1 BPE tokenizer on `train_text` and return the trained tokenizer."""
    tokenizer = ByteLevelBPE(vocab_size)
    tokenizer.fit(train_text)
    return tokenizer

## 9. Define The Model

## LMConfig

In [5]:
@dataclass
class LMConfig:
    # ---- general ----
    vocab_size: int = VOCAB_SIZE
    max_seq_len: int = MODEL_CONTEXT_LENGTH
    d_model: int = N_EMBD
    n_layers: int = N_LAYER
    dropout: float = 0.0
    bias: bool = False
    tie_embeddings: bool = True
    norm_type: str = "rmsnorm"          # "rmsnorm" | "layernorm"
    norm_eps: float = 1e-6

    # ---- positional ----
    pos_emb_type: str = "rope"          # "rope" | "sinusoidal"
    rope_theta: float = 10000.0

    # ---- attention ----
    attn_type: str = "gqa"              # "gqa" | "mla"
    n_heads: int = N_HEAD
    # GQA (n_kv_heads == n_heads -> MHA, 1 -> MQA)
    n_kv_heads: int = 2
    head_dim: Optional[int] = None      # defaults to d_model // n_heads
    # MLA
    q_lora_rank: Optional[int] = 256    # None -> no query compression
    kv_lora_rank: int = 128
    qk_nope_head_dim: int = 48
    qk_rope_head_dim: int = 16
    v_head_dim: int = 64

    # ---- feed-forward ----
    ffn_type: str = "dense"             # "dense" | "moe"
    d_ff: int = 2048
    # MoE
    n_experts: int = 8
    top_k: int = 2
    n_shared_experts: int = 1
    expert_d_ff: int = 512
    aux_loss_coef: float = 0.01
    first_k_dense_layers: int = 0       # first k layers stay dense
    moe_layer_freq: int = 1             # after that, every n-th layer is MoE

    def __post_init__(self):
        assert self.norm_type in ("rmsnorm", "layernorm")
        assert self.pos_emb_type in ("rope", "sinusoidal")
        assert self.attn_type in ("gqa", "mla")
        assert self.ffn_type in ("dense", "moe")

        if self.pos_emb_type == "sinusoidal":
            assert self.d_model % 2 == 0, "sinusoidal needs even d_model"

        if self.attn_type == "gqa":
            if self.head_dim is None:
                assert self.d_model % self.n_heads == 0, "d_model must be divisible by n_heads"
                self.head_dim = self.d_model // self.n_heads
            assert self.n_heads % self.n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
            if self.pos_emb_type == "rope":
                assert self.head_dim % 2 == 0, "RoPE needs even head_dim"
        else:
            if self.pos_emb_type == "rope":
                assert self.qk_rope_head_dim % 2 == 0 and self.qk_rope_head_dim > 0

        if self.ffn_type == "moe":
            assert 1 <= self.top_k <= self.n_experts

    @property
    def rope_dim(self) -> int:
        return self.qk_rope_head_dim if self.attn_type == "mla" else self.head_dim

    def is_moe_layer(self, layer_idx: int) -> bool:
        if self.ffn_type != "moe" or layer_idx < self.first_k_dense_layers:
            return False
        return (layer_idx - self.first_k_dense_layers) % self.moe_layer_freq == 0

## Normalization layers

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        dtype = x.dtype
        x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * self.weight.float()).to(dtype)


def make_norm(cfg: LMConfig, dim: int) -> nn.Module:
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(dim, cfg.norm_eps)
    return nn.LayerNorm(dim, eps=cfg.norm_eps)

## Positional embeddings (RoPE and sinusoidal)

In [7]:
class RoPE(nn.Module):
    """Rotary embedding. Rotates q and k of shape (B, H, T, dim)."""

    def __init__(self, dim: int, max_seq_len: int, theta: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)               # (L, dim/2)
        emb = torch.cat([freqs, freqs], dim=-1)        # (L, dim)
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    @staticmethod
    def _rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1)
        return torch.cat([-x2, x1], dim=-1)

    def rotate(self, x, positions):
        cos, sin = self.cos[positions], self.sin[positions]
        if cos.dim() == 2:                              # positions: (T,)
            cos, sin = cos[None, None], sin[None, None]
        else:                                           # positions: (B, T)
            cos, sin = cos[:, None], sin[:, None]
        cos, sin = cos.to(x.dtype), sin.to(x.dtype)
        return x * cos + self._rotate_half(x) * sin

    def forward(self, q, k, positions):
        return self.rotate(q, positions), self.rotate(k, positions)


class SinusoidalPositionalEmbedding(nn.Module):
    """Fixed sin/cos table, added to token embeddings once at model level."""

    def __init__(self, d_model: int, max_seq_len: int):
        super().__init__()
        pos = torch.arange(max_seq_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_seq_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x, offset: int = 0):
        T = x.size(1)
        return x + self.pe[offset:offset + T].to(x.dtype)

## Attention helper and GQA

In [8]:
def causal_sdpa(q, k, v, dropout_p: float = 0.0, training: bool = False):
    """Causal attention that also works when k/v are longer than q (KV cache)."""
    Tq, Tk = q.size(2), k.size(2)
    p = dropout_p if training else 0.0
    if Tq == Tk:
        return F.scaled_dot_product_attention(q, k, v, dropout_p=p, is_causal=True)
    if Tq == 1:
        return F.scaled_dot_product_attention(q, k, v, dropout_p=p)
    mask = torch.ones(Tq, Tk, dtype=torch.bool, device=q.device).tril(diagonal=Tk - Tq)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p)


def masked_sdpa(q, k, v, attn_mask, dropout_p=0.0, training=False):
    if attn_mask is None:
        return causal_sdpa(q, k, v, dropout_p, training)            # training path unchanged
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                          dropout_p=dropout_p if training else 0.0)

class GQAttention(nn.Module):
    """Grouped-query attention. cache = (k, v), each (B, n_kv_heads, T, head_dim)."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.n_heads, self.n_kv, self.hd = cfg.n_heads, cfg.n_kv_heads, cfg.head_dim
        self.q_proj = nn.Linear(cfg.d_model, self.n_heads * self.hd, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, self.n_kv * self.hd, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, self.n_kv * self.hd, bias=cfg.bias)
        self.o_proj = nn.Linear(self.n_heads * self.hd, cfg.d_model, bias=cfg.bias)
        self.dropout = cfg.dropout

    def forward(self, x, rope=None, positions=None, kv_cache=None, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv, self.hd).transpose(1, 2)
        if rope is not None:
            q, k = rope(q, k, positions)
        if kv_cache is not None:
            k = torch.cat([kv_cache[0], k], dim=2)
            v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = (k, v)
        rep = self.n_heads // self.n_kv
        out = masked_sdpa(q, k.repeat_interleave(rep, dim=1), v.repeat_interleave(rep, dim=1),
                          attn_mask, self.dropout, self.training)
        out = out.transpose(1, 2).reshape(B, T, self.n_heads * self.hd)
        return self.o_proj(out), new_cache

## MLA

In [9]:
class MLAttention(nn.Module):
    """
    Multi-head Latent Attention (DeepSeek-style, un-absorbed version).
    cache = (latent (B, 1, T, kv_lora_rank), k_rope (B, 1, T, rope_dim) or None).
    Only the compressed latent and the small shared k_rope are cached.
    With sinusoidal positions the decoupled RoPE part is dropped (rope_dim = 0).
    """

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.H = cfg.n_heads
        self.dn = cfg.qk_nope_head_dim
        self.dr = cfg.qk_rope_head_dim if cfg.pos_emb_type == "rope" else 0
        self.dv = cfg.v_head_dim
        self.dropout = cfg.dropout

        q_out = self.H * (self.dn + self.dr)
        if cfg.q_lora_rank:
            self.q_down = nn.Linear(cfg.d_model, cfg.q_lora_rank, bias=cfg.bias)
            self.q_norm = make_norm(cfg, cfg.q_lora_rank)
            self.q_up = nn.Linear(cfg.q_lora_rank, q_out, bias=False)
            self.q_proj = None
        else:
            self.q_proj = nn.Linear(cfg.d_model, q_out, bias=cfg.bias)

        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora_rank, bias=cfg.bias)
        self.kv_norm = make_norm(cfg, cfg.kv_lora_rank)
        self.kv_up = nn.Linear(cfg.kv_lora_rank, self.H * (self.dn + self.dv), bias=False)
        self.k_rope_proj = nn.Linear(cfg.d_model, self.dr, bias=cfg.bias) if self.dr > 0 else None
        self.o_proj = nn.Linear(self.H * self.dv, cfg.d_model, bias=cfg.bias)

    def forward(self, x, rope=None, positions=None, kv_cache=None, attn_mask=None):
        B, T, _ = x.shape
        q = self.q_up(self.q_norm(self.q_down(x))) if self.q_proj is None else self.q_proj(x)
        q = q.view(B, T, self.H, self.dn + self.dr).transpose(1, 2)
        latent = self.kv_norm(self.kv_down(x)).unsqueeze(1)
        k_rope = None
        if self.dr > 0:
            q_nope, q_rope = q.split([self.dn, self.dr], dim=-1)
            k_rope = self.k_rope_proj(x).unsqueeze(1)
            q_rope, k_rope = rope(q_rope, k_rope, positions)
        else:
            q_nope = q
        if kv_cache is not None:
            latent = torch.cat([kv_cache[0], latent], dim=2)
            if k_rope is not None:
                k_rope = torch.cat([kv_cache[1], k_rope], dim=2)
        new_cache = (latent, k_rope)
        Tk = latent.size(2)
        kv = self.kv_up(latent.squeeze(1)).view(B, Tk, self.H, self.dn + self.dv).transpose(1, 2)
        k_nope, v = kv.split([self.dn, self.dv], dim=-1)
        if self.dr > 0:
            q_full = torch.cat([q_nope, q_rope], dim=-1)
            k_full = torch.cat([k_nope, k_rope.expand(-1, self.H, -1, -1)], dim=-1)
        else:
            q_full, k_full = q_nope, k_nope
        out = masked_sdpa(q_full, k_full, v, attn_mask, self.dropout, self.training)
        out = out.transpose(1, 2).reshape(B, T, self.H * self.dv)
        return self.o_proj(out), new_cache


ATTN_REGISTRY = {"gqa": GQAttention, "mla": MLAttention}

## Feed-forward networks (GeLU)

In [10]:
class DenseFFN(nn.Module):
    """Linear -> GeLU -> Linear. Returns (out, aux_loss=0)."""

    def __init__(self, cfg: LMConfig, d_ff: Optional[int] = None):
        super().__init__()
        d_ff = d_ff or cfg.d_ff
        self.fc1 = nn.Linear(cfg.d_model, d_ff, bias=cfg.bias)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(d_ff, cfg.d_model, bias=cfg.bias)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x))), x.new_zeros(())


class MoEFFN(nn.Module):
    """Top-k routed experts (DenseFFN each) + optional always-on shared experts."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.E, self.k = cfg.n_experts, cfg.top_k
        self.router = nn.Linear(cfg.d_model, self.E, bias=False)
        self.experts = nn.ModuleList([DenseFFN(cfg, cfg.expert_d_ff) for _ in range(self.E)])
        self.shared = nn.ModuleList([DenseFFN(cfg, cfg.expert_d_ff) for _ in range(cfg.n_shared_experts)])

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                  # (N, D)

        probs = self.router(flat).float().softmax(dim=-1)        # (N, E)
        topv, topi = probs.topk(self.k, dim=-1)                  # (N, k)
        topv = topv / topv.sum(dim=-1, keepdim=True)             # renormalise

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.experts):
            tok_idx, slot = (topi == e).nonzero(as_tuple=True)
            if tok_idx.numel() == 0:
                continue
            y, _ = expert(flat[tok_idx])
            w = topv[tok_idx, slot].unsqueeze(-1).to(y.dtype)
            out.index_add_(0, tok_idx, (y * w).to(out.dtype))

        for sh in self.shared:
            out = out + sh(flat)[0]

        # Switch-style load-balancing loss: E * sum_i f_i * P_i
        if self.training:
            f = F.one_hot(topi, self.E).sum(dim=1).float().mean(dim=0) / self.k
            P = probs.mean(dim=0)
            aux = self.E * (f * P).sum()
        else:
            aux = x.new_zeros(())
        return out.view(B, T, D), aux


FFN_REGISTRY = {"dense": DenseFFN, "moe": MoEFFN}

## TransformerBlock

In [11]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: LMConfig, layer_idx: int):
        super().__init__()
        self.norm1 = make_norm(cfg, cfg.d_model)
        self.attn = ATTN_REGISTRY[cfg.attn_type](cfg)
        self.norm2 = make_norm(cfg, cfg.d_model)
        self.ffn = FFN_REGISTRY["moe" if cfg.is_moe_layer(layer_idx) else "dense"](cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, positions=None, kv_cache=None, attn_mask=None):
        a, new_cache = self.attn(self.norm1(x), rope, positions, kv_cache, attn_mask)
        x = x + self.drop(a)
        f, aux = self.ffn(self.norm2(x))
        x = x + self.drop(f)
        return x, aux, new_cache

## DecoderLM

In [12]:
class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.sin_emb = (SinusoidalPositionalEmbedding(cfg.d_model, cfg.max_seq_len)
                        if cfg.pos_emb_type == "sinusoidal" else None)
        self.rope = (RoPE(cfg.rope_dim, cfg.max_seq_len, cfg.rope_theta)
                     if cfg.pos_emb_type == "rope" else None)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([TransformerBlock(cfg, i) for i in range(cfg.n_layers)])
        self.final_norm = make_norm(cfg, cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)
        # scaled init on residual output projections
        for name, p in self.named_parameters():
            if name.endswith("o_proj.weight") or name.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, targets=None, kv_caches=None, use_cache=False,
                last_only=False, attention_mask=None):
        B, T = input_ids.shape
        past = 0 if kv_caches is None else kv_caches[0][0].size(2)
        dev = input_ids.device
        if attention_mask is None:                                      # training / unpadded: unchanged behaviour
            assert past + T <= self.cfg.max_seq_len, "sequence exceeds max_seq_len"
            positions = torch.arange(past, past + T, device=dev)        # (T,)
            attn_mask = None
        else:
            am = attention_mask.to(dev).bool()
            assert am.shape == (B, past + T), f"attention_mask must be (B, past+T) = {(B, past + T)}"
            positions = (am.long().cumsum(-1) - 1).clamp(min=0)[:, past:]            # (B, T) per-row positions
            q_idx = torch.arange(past, past + T, device=dev)[:, None]
            k_idx = torch.arange(past + T, device=dev)[None, :]
            # causal AND key is a real token; the diagonal is always allowed so padded query rows never become all -inf
            attn_mask = ((k_idx <= q_idx)[None, None] & am[:, None, None, :]) | (k_idx == q_idx)[None, None]
    
        x = self.tok_emb(input_ids)
        if self.sin_emb is not None:
            x = x * math.sqrt(self.cfg.d_model) + self.sin_emb.pe[positions].to(x.dtype)
        x = self.drop(x)
    
        aux_total = x.new_zeros(())
        new_caches = []
        for i, block in enumerate(self.blocks):
            cache_i = None if kv_caches is None else kv_caches[i]
            x, aux, new_cache = block(x, self.rope, positions, cache_i, attn_mask)
            aux_total = aux_total + aux
            new_caches.append(new_cache)
    
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), targets.reshape(-1), ignore_index=-100)
            loss = ce + self.cfg.aux_loss_coef * aux_total
        caches = new_caches if (use_cache or kv_caches is not None) else None
        return logits, loss, caches

    @torch.no_grad()
    def generate(self, input_ids, attention_mask=None, max_new_tokens=400, temperature=0.4, top_k=50, top_p=0.9,
                 repetition_penalty=1.1, rep_window=256, no_repeat_ngram=0, ban_ids=(), pad_id=0, 
                 tokenizer=None, target_words=None, check_every=16):
        """input_ids: (B, L) LEFT-padded; attention_mask: (B, L) with 1 for real tokens, 0 for padding."""
        was_training = self.training
        self.eval()
        B, L = input_ids.shape
        dev = input_ids.device
        attn = (torch.ones(B, L, dtype=torch.long, device=dev) if attention_mask is None
                else attention_mask.to(dev).long())
        assert int(attn.sum(1).max()) + max_new_tokens <= self.cfg.max_seq_len, \
            "longest prompt + new tokens exceed max_seq_len"
        ban = torch.tensor(sorted(set(ban_ids)), dtype=torch.long, device=dev) if len(ban_ids) else None
    
        n = no_repeat_ngram
        seen = [dict() for _ in range(B)]
        hist = [[] for _ in range(B)]
    
        gen = input_ids.new_zeros((B, 0))
        done = torch.zeros(B, dtype=torch.bool, device=dev)
        caches, cur = None, input_ids
        for step in range(max_new_tokens):
            logits, _, caches = self(cur, kv_caches=caches, use_cache=True, last_only=True, attention_mask=attn)
            logits = logits[:, -1, :].float()
            if ban is not None:
                logits[:, ban] = float("-inf")
            if repetition_penalty != 1.0 and gen.size(1) > 0:
                recent = gen[:, -rep_window:]
                sc = torch.gather(logits, 1, recent)
                sc = torch.where(sc > 0, sc / repetition_penalty, sc * repetition_penalty)
                logits.scatter_(1, recent, sc)
            if n > 1:
                for b in range(B):
                    if len(hist[b]) >= n - 1:
                        banned = seen[b].get(tuple(hist[b][-(n - 1):]))
                        if banned:
                            logits[b, list(banned)] = float("-inf")
    
            if temperature <= 0:
                nxt = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1, None]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                if top_p and top_p < 1.0:
                    sl, si = torch.sort(logits, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    sl = sl.masked_fill(pr.cumsum(-1) - pr > top_p, float("-inf"))
                    logits = torch.full_like(logits, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
    
            nxt = torch.where(done.unsqueeze(1), torch.full_like(nxt, pad_id), nxt)
            if n > 1:
                for b, t in enumerate(nxt.squeeze(1).tolist()):
                    h = hist[b]
                    if len(h) >= n - 1:
                        seen[b].setdefault(tuple(h[-(n - 1):]), set()).add(t)
                    h.append(t)
            gen = torch.cat([gen, nxt], dim=1)
            cur = nxt
            attn = torch.cat([attn, torch.ones(B, 1, dtype=attn.dtype, device=dev)], dim=1)   # new token is a real token
    
            if target_words is not None and (step + 1) % check_every == 0:
                for b in torch.nonzero(~done).flatten().tolist():
                    if len(tokenizer.decode(gen[b].tolist()).split()) > target_words:
                        done[b] = True
                if done.all():
                    break
        self.train(was_training)
        return torch.cat([input_ids, gen], dim=1)

In [1]:
!zip -r best_models.zip checkpoints -i "*_best.pt"

  adding: checkpoints/mla-rope-moe_best.pt (deflated 7%)
  adding: checkpoints/gqa-sinusoidal-moe_best.pt (deflated 7%)
  adding: checkpoints/abl_mla_rope_moe_best.pt (deflated 7%)
  adding: checkpoints/mla-sinusoidal-moe_best.pt (deflated 7%)
  adding: checkpoints/gqa-rope-moe_best.pt (deflated 7%)
  adding: checkpoints/abl_gqa_sin_moe_best.pt (deflated 7%)
  adding: checkpoints/abl_gqa_rope_moe_best.pt (deflated 7%)
  adding: checkpoints/abl_mla_sin_moe_best.pt (deflated 7%)


# Phase 2 - L-NER

In [3]:
!cp -rv ../input/datasets/pritam0289/phase1-models/ ./ 

'../input/datasets/pritam0289/phase1-models/' -> './phase1-models'
'../input/datasets/pritam0289/phase1-models/tokenizer.pkl' -> './phase1-models/tokenizer.pkl'
'../input/datasets/pritam0289/phase1-models/best_models' -> './phase1-models/best_models'
'../input/datasets/pritam0289/phase1-models/best_models/abl_gqa_rope_moe_best.pt' -> './phase1-models/best_models/abl_gqa_rope_moe_best.pt'
'../input/datasets/pritam0289/phase1-models/best_models/abl_gqa_sin_moe_best.pt' -> './phase1-models/best_models/abl_gqa_sin_moe_best.pt'
'../input/datasets/pritam0289/phase1-models/best_models/abl_mla_sin_moe_best.pt' -> './phase1-models/best_models/abl_mla_sin_moe_best.pt'
'../input/datasets/pritam0289/phase1-models/best_models/abl_mla_rope_moe_best.pt' -> './phase1-models/best_models/abl_mla_rope_moe_best.pt'


In [4]:
!ls

phase1-models  state.db


In [13]:
with open("./phase1-models/tokenizer.pkl", "rb") as f:
        tokenizer = pickle.load(f)

In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


def get_device():
    """
      Return the computing device used in the current session.
    """
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = get_device()
print("Using device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Using device: cuda
Tesla T4


In [75]:
import os, re, json, math, time, copy, random, zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

# ---------------------------- configuration ----------------------------

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
NUM_WORKERS = 3
# DEVICE = globals().get(
#     "DEVICE", torch.device("cuda" if torch.cuda.is_available() else "cpu")
# )

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    
HF_REPO = globals().get(
    "HF_REPO", globals().get("HF_DATASET_REPO", "Exploration-Lab/CS-781-Capstone")
)
HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN"))

WIN = 512
STRIDE = 256
BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
EPOCHS = {
    "full_ft": 13,
    "partial_top3": 10,
    "lora_r8": 10,
    "lorma_r8": 10,
    "adapter64": 10,
    "kd_student": 10,
    "student_noKD": 10,
}
LR = {
    "full_ft": 5e-5,
    "partial_top3": 1e-4,
    "lora_r8": 1e-3,
    "lorma_r8": 1e-3,
    "adapter64": 1e-3,
    "kd_student": 1e-4,
    "student_noKD": 1e-4,
}
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
CAUSAL = False  # False: bidirectional NER attention; True: retain causal attention
KD_T = 2.0
KD_ALPHA = 0.5
ADAPTER_DIM = 64
LORA_R = 8
TOP_K_BLOCKS = 3
KD_KEEP_BLOCKS = [0, 2, 4, 7]

PHASE1_CKPT_DIR = "/kaggle/working/phase1-models/best_models"
OUT = "/kaggle/working/lner_runs"
os.makedirs(OUT, exist_ok=True)

VARIANTS = [
    "gqa_rope_moe",
    "mla_rope_moe",
    "gqa_sin_moe",
    "mla_sin_moe",
]
BASE = dict(
    vocab_size=30000, max_seq_len=2048, d_model=768, n_layers=16, n_heads=8, n_kv_heads=2,
    dropout=0.0, bias=False, tie_embeddings=True, d_ff=2048,
    q_lora_rank=384, kv_lora_rank=192, qk_nope_head_dim=80, qk_rope_head_dim=16, v_head_dim=96,
)
MOE = dict(ffn_type="moe", first_k_dense_layers=2, n_experts=8, top_k=2, n_shared_experts=1,
           expert_d_ff=512, aux_loss_coef=0.01)

CONFIGS = {
    "gqa_rope_moe": LMConfig(**BASE, **MOE, attn_type="gqa", pos_emb_type="rope"),
    "mla_rope_moe": LMConfig(**BASE, **MOE, attn_type="mla", pos_emb_type="rope"),
    "gqa_sin_moe":  LMConfig(**BASE, **MOE, attn_type="gqa", pos_emb_type="sinusoidal"),
    "mla_sin_moe":  LMConfig(**BASE, **MOE, attn_type="mla", pos_emb_type="sinusoidal"),
}

assert "CONFIGS" in globals(), "Run the Phase-1 CONFIGS cell first."
assert "DecoderLM" in globals(), "Run the Phase-1 model-definition cells first."
assert "tokenizer" in globals(), "Load the trained Phase-1 tokenizer first."
assert all(v in CONFIGS for v in VARIANTS), "One or more expected variants are missing from CONFIGS."

# AMP selection compatible with Kaggle T4/A100 and CPU.
USE_AMP = DEVICE.type == "cuda"
if USE_AMP:
    AMP_DTYPE = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
else:
    AMP_DTYPE = torch.float32
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16

def autocast_context():
    return torch.autocast(
        device_type=DEVICE.type,
        dtype=AMP_DTYPE,
        enabled=USE_AMP
    )

USE_AMP = DEVICE.type == "cuda"

if USE_AMP:
    AMP_DTYPE = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
else:
    AMP_DTYPE = torch.float32

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16

print("Device:", DEVICE, "| AMP:", USE_AMP, AMP_DTYPE)

print("Device:", DEVICE, "| AMP:", USE_AMP, AMP_DTYPE)
print("Phase-1 checkpoint directory:", PHASE1_CKPT_DIR)
print("Phase-2 output directory:", OUT)

GPU: Tesla T4
GPU memory (GB): 14.56
Device: cuda | AMP: True torch.bfloat16
Device: cuda | AMP: True torch.bfloat16
Phase-1 checkpoint directory: /kaggle/working/phase1-models/best_models
Phase-2 output directory: /kaggle/working/lner_runs


In [16]:

# ---------------------------- model/tokenizer loaders ----------------------------

def load_tokenizer():
    """Uses the tokenizer object loaded by Phase 1; never retrains it."""
    tok = globals().get("tokenizer")
    if tok is None:
        raise RuntimeError("Phase-1 tokenizer is not loaded.")
    for attr in ("pat", "ranks", "merged_map", "stoi", "itos", "encode", "decode"):
        if not hasattr(tok, attr):
            raise TypeError(f"Phase-1 tokenizer lacks required attribute {attr!r}.")
    return tok

tokenizer = load_tokenizer()

def phase1_checkpoint_path(variant):
    # Phase-1 Cell 41 saves checkpoints/<tag>.pt.
    candidates = [
        os.path.join(PHASE1_CKPT_DIR, f"abl_{variant}_best.pt"),
        os.path.join(PHASE1_CKPT_DIR, f"{variant}_best.pt"),
        os.path.join(PHASE1_CKPT_DIR, f"abl_{variant}.pt"),
        os.path.join(PHASE1_CKPT_DIR, f"{variant}.pt"),
        os.path.join("/kaggle/working", PHASE1_CKPT_DIR, f"abl_{variant}_best.pt"),
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(
        f"No Phase-1 checkpoint found for {variant}. Checked:\n" + "\n".join(candidates)
        + "\nSet PHASE1_CKPT_DIR to the directory containing your saved Phase-1 .pt files."
    )

def load_phase1_model(variant, checkpoint_path=None):
    """Rebuild the exact Phase-1 architecture and load its trained weights."""
    if variant not in CONFIGS:
        raise ValueError(f"Unknown variant {variant}; available: {list(CONFIGS)}")
    path = checkpoint_path or phase1_checkpoint_path(variant)
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model = DecoderLM(CONFIGS[variant])
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=True)
    model.to(DEVICE)
    print(f"Loaded Phase-1 {variant} from {path}")
    return model

In [17]:

# ---------------------------- dataset loading ----------------------------

def dl(name):
    if not HF_TOKEN:
        raise RuntimeError(
            "HF_TOKEN is not available. Run your Phase-1 Hugging Face login cell first."
        )
    return hf_hub_download(
        repo_id=HF_REPO,
        filename=f"Phase_2_LNER/{name}",
        repo_type="dataset",
        token=HF_TOKEN
    )

with open(dl("Train/labels.txt"), encoding="utf-8") as f:
    LABELS = [line.strip() for line in f if line.strip()]

BIO = ["O"] + [f"{p}-{label}" for label in LABELS for p in ("B", "I")]
L2I = {label: i for i, label in enumerate(BIO)}
I2L = {i: label for label, i in L2I.items()}
NL = len(BIO)

def load_fold(k):
    with open(dl(f"Train/fold{k}-text.json"), encoding="utf-8") as f:
        texts = json.load(f)
    with open(dl(f"Train/fold{k}-gold.json"), encoding="utf-8") as f:
        gold = json.load(f)

    text_by_id = {str(k): v for k, v in texts.items()}
    entities = {}
    for doc_id in text_by_id:
        rec = gold.get(doc_id, gold.get(int(doc_id)) if doc_id.isdigit() else None)
        if isinstance(rec, dict):
            rec = rec.get("entities", [])
        parsed = []
        for e in rec or []:
            if isinstance(e, dict):
                start = e.get("start", e.get("start_offset"))
                end = e.get("end", e.get("end_offset"))
                label = e.get("label", e.get("type"))
                text_value = e.get("text")
            else:
                start, end, label = e[:3]
                text_value = e[3] if len(e) > 3 else None
            parsed.append((int(start), int(end), label, text_value))
        entities[doc_id] = parsed
    return text_by_id, entities

train_texts, train_entities = load_fold(1)
val_texts, val_entities = load_fold(2)


print("Labels:", LABELS)
print("Train documents:", len(train_texts), "| validation documents:", len(val_texts))


labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Train documents: 35 | validation documents: 35


In [18]:
# ---------------------------- tokenizer offsets / BIO alignment ----------------------------
# Use the labels loaded from your dataset
BIO = ["O"] + [
    f"{prefix}-{label}"
    for label in LABELS
    for prefix in ("B", "I")
]

l2i = {label: idx for idx, label in enumerate(BIO)}
NL = len(BIO)

print("Number of BIO classes:", NL)
print("Example mappings:", {k: l2i[k] for k in BIO[:5]})
def encode_with_offsets(text, tok=tokenizer):
    """
    Encode text using the existing BPE tokenizer and return:
      ids: token IDs
      offsets: (start, end) character offsets into the original text

    Offsets use Python's [start, end) convention.
    """
    ids = []
    offsets = []

    for match in tok.pat.finditer(text):
        word = match.group()
        if not word:
            continue

        # Split the word into UTF-8 bytes, matching the tokenizer's BPE.
        word_bytes = word.encode("utf-8")
        pieces = [
            (bytes([b]), i, i + 1)
            for i, b in enumerate(word_bytes)
        ]

        # Map every UTF-8 byte to its original character index.
        byte_to_char = []
        for char_i, char in enumerate(word):
            byte_to_char.extend([char_i] * len(char.encode("utf-8")))

        # Apply the same ranked BPE merge procedure as the tokenizer.
        while len(pieces) > 1:
            best_i = -1
            best_rank = float("inf")

            for i in range(len(pieces) - 1):
                pair = (pieces[i][0], pieces[i + 1][0])
                rank = tok.ranks.get(pair, float("inf"))

                if rank < best_rank:
                    best_i = i
                    best_rank = rank

            if best_i == -1:
                break

            left = pieces[best_i]
            right = pieces[best_i + 1]

            pieces[best_i:best_i + 2] = [
                (left[0] + right[0], left[1], right[2])
            ]

        # Convert each merged piece's byte range to character offsets.
        for piece, byte_start, byte_end in pieces:
            start = match.start() + byte_to_char[byte_start]
            end = match.start() + byte_to_char[byte_end - 1] + 1

            # Remove leading whitespace from the token's character span,
            # while preserving the original text's coordinate system.
            while start < end and text[start].isspace():
                start += 1

            token_id = tok.stoi.get(piece, tok.stoi["<unk>"])

            ids.append(token_id)
            offsets.append((start, end))

    return ids, offsets

def token_labels(entities, offsets):
    """
    entities: [(start, end, label, text), ...]
    offsets: [(start, end), ...]

    Returns:
      y: BIO label ID for each token
      exact: number of entities with exact token boundaries
    """
    y = [l2i["O"]] * len(offsets)
    exact = 0

    for s, e, label, _ in entities:
        if not (0 <= s < e):
            continue

        overlapping = [
            i for i, (a, b) in enumerate(offsets)
            if a < b and a < e and b > s
        ]

        if not overlapping:
            continue

        first = overlapping[0]
        last = overlapping[-1]

        if offsets[first][0] == s and offsets[last][1] == e:
            exact += 1

        # Assign BIO labels to the overlapping tokens.
        y[first] = l2i[f"B-{label}"]

        for i in overlapping[1:]:
            y[i] = l2i[f"I-{label}"]

    return y, exact

def build_docs(texts, entities, desc):
    docs = {}
    for doc_id, text in tqdm(texts.items(), desc=desc):
        ids, offsets = encode_with_offsets(text)
        tags, exact = token_labels(entities[doc_id], offsets)
        docs[doc_id] = {
            "text": text, "ids": ids, "offs": offsets, "y": tags,
            "gold": [(s, e, lab) for s, e, lab, _ in entities[doc_id]],
            "exact": exact
        }
    return docs

train_docs = build_docs(train_texts, train_entities, "Encode train")
val_docs = build_docs(val_texts, val_entities, "Encode validation")
n_gold_train = sum(len(x["gold"]) for x in train_docs.values())
n_exact_train = sum(x["exact"] for x in train_docs.values())
print(f"Exact token-boundary entities (train): {n_exact_train}/{n_gold_train}")



Number of BIO classes: 25
Example mappings: {'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4}


Encode train:   0%|          | 0/35 [00:00<?, ?it/s]

Encode validation:   0%|          | 0/35 [00:00<?, ?it/s]

Exact token-boundary entities (train): 4049/4120


In [19]:
from sklearn.model_selection import train_test_split

# 1. Merge training and validation documents
all_docs = {**train_docs, **val_docs}

# 2. Extract gold annotations directly from the merged documents
all_gold = {
    doc_id: doc["gold"]
    for doc_id, doc in all_docs.items()
}

# 3. Split by document ID
doc_ids = list(all_docs.keys())

train_ids, val_ids = train_test_split(
    doc_ids,
    test_size=0.2,
    random_state=SEED
)

# 4. Create the new datasets
train_docs = {doc_id: all_docs[doc_id] for doc_id in train_ids}
val_docs = {doc_id: all_docs[doc_id] for doc_id in val_ids}

# 5. Update validation gold annotations
gold_v = {doc_id: all_gold[doc_id] for doc_id in val_ids}

print(f"Training documents: {len(train_docs)}")
print(f"Validation documents: {len(val_docs)}")
print(f"Total documents: {len(train_docs) + len(val_docs)}")

Training documents: 56
Validation documents: 14
Total documents: 70


In [50]:

# ---------------------------- BIO decoding and strict metrics ----------------------------

def viterbi(logp):
    T = logp.shape[0]
    if T == 0:
        return []
    allow = np.zeros((NL, NL), dtype=bool)
    for prev in range(NL):
        for cur in range(NL):
            name = BIO[cur]
            allow[prev, cur] = not (
                name.startswith("I-")
                and not (BIO[prev] != "O" and BIO[prev][2:] == name[2:])
            )
    trans = np.where(allow, 0.0, -1e9)
    start = np.array([0.0 if not BIO[c].startswith("I-") else -1e9 for c in range(NL)])
    dp = start + logp[0]
    back = np.zeros((T, NL), dtype=np.int64)
    for t in range(1, T):
        scores = dp[:, None] + trans
        back[t] = scores.argmax(axis=0)
        dp = scores.max(axis=0) + logp[t]
    path = [int(dp.argmax())]
    for t in range(T - 1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return path[::-1]

def tags_to_spans(tags, offsets):
    spans, current = [], None
    for tag_id, (start, end) in zip(tags, offsets):
        if start == end:
            continue
        name = BIO[int(tag_id)]
        if name == "O":
            if current is not None:
                spans.append(tuple(current))
                current = None
        elif name.startswith("B-") or current is None or current[2] != name[2:]:
            if current is not None:
                spans.append(tuple(current))
            current = [start, end, name[2:]]
        else:
            current[1] = end
    if current is not None:
        spans.append(tuple(current))
    return spans

def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc_id in set(gold) | set(pred):
        G = set(tuple(x) for x in gold.get(doc_id, []))
        P = set(tuple(x) for x in pred.get(doc_id, []))
        for ent in P:
            (tp if ent in G else fp)[ent[2]] += 1
        for ent in G - P:
            fn[ent[2]] += 1
    per = {}
    for label in LABELS:
        if tp[label] + fp[label] + fn[label] == 0:
            continue
        p = tp[label] / max(1, tp[label] + fp[label])
        r = tp[label] / max(1, tp[label] + fn[label])
        f = 2 * p * r / max(1e-12, p + r)
        per[label] = (p, r, f, tp[label] + fn[label])
    mean = lambda i: float(np.mean([v[i] for v in per.values()])) if per else 0.0
    return {"P": mean(0), "R": mean(1), "F1": mean(2), "per_label": per}

def error_analysis(gold, pred):
    counts = Counter()
    for doc_id in set(gold) | set(pred):
        G = set(tuple(x) for x in gold.get(doc_id, []))
        P = set(tuple(x) for x in pred.get(doc_id, []))
        for x in P:
            if x in G:
                counts["exact"] += 1
            elif any((g[0], g[1]) == (x[0], x[1]) for g in G):
                counts["label_confusion"] += 1
            elif any(g[2] == x[2] and g[0] < x[1] and x[0] < g[1] for g in G):
                counts["boundary_error"] += 1
            else:
                counts["false_positive"] += 1
        for g in G - P:
            if not any(p[0] < g[1] and g[0] < p[1] for p in P):
                counts["missed(FN)"] += 1
    return dict(counts)

# def show(res, title=""):
#     print(f"{title} | macro P {res['P']:.4f} R {res['R']:.4f} F1 {res['F1']:.4f}")
#     for label, (p, r, f, n) in res["per_label"].items():
#         print(f"  {label:15s} P {p:.3f} R {r:.3f} F1 {f:.3f} n={n}")
def show(res, title):
    print(
        f"{title} | "
        f"macro P {float(res['P']):.4f} "
        f"R {float(res['R']):.4f} "
        f"F1 {float(res['F1']):.4f}"
    )

    # Support both possible per-class result formats.
    per_label = res.get("per_label", res.get("per", {}))

    for label, values in per_label.items():
        # Expected format: (precision, recall, F1, support)
        if isinstance(values, dict):
            p = values.get("P", values.get("precision"))
            r = values.get("R", values.get("recall"))
            f1 = values.get("F1", values.get("f1"))
            n = values.get("n", values.get("support", 0))
        else:
            p, r, f1, n = values

        print(
            f"  {label:15s} "
            f"P {float(p):.3f} "
            f"R {float(r):.3f} "
            f"F1 {float(f1):.3f} "
            f"n={n}"
        )

gold_v = {doc_id: x["gold"] for doc_id, x in val_docs.items()}
oracle = {doc_id: tags_to_spans(x["y"], x["offs"]) for doc_id, x in val_docs.items()}
show(strict_prf(gold_v, oracle), "ORACLE (token alignment upper bound)")


ORACLE (token alignment upper bound) | macro P 0.9777 R 0.9777 F1 0.9777
  APP             P 0.934 R 0.934 F1 0.934 n=61
  RESP            P 1.000 R 1.000 F1 1.000 n=66
  A.COUNSEL       P 1.000 R 1.000 F1 1.000 n=26
  R.COUNSEL       P 1.000 R 1.000 F1 1.000 n=21
  JUDGE           P 1.000 R 1.000 F1 1.000 n=64
  WIT             P 1.000 R 1.000 F1 1.000 n=62
  AUTH            P 0.974 R 0.974 F1 0.974 n=190
  COURT           P 1.000 R 1.000 F1 1.000 n=136
  STAT            P 0.948 R 0.948 F1 0.948 n=212
  PREC            P 0.876 R 0.876 F1 0.876 n=201
  DATE            P 1.000 R 1.000 F1 1.000 n=315
  CASENO          P 1.000 R 1.000 F1 1.000 n=161


In [38]:
def decode_bio(tag_ids, offsets):
    spans = []
    current = None

    for tag_id, (start, end) in zip(tag_ids, offsets):
        if start == end:
            continue

        tag = id_to_label[int(tag_id)]

        if tag == "O":
            if current is not None:
                spans.append(tuple(current))
                current = None
            continue

        prefix, label = tag.split("-", 1)

        if (
            prefix == "B"
            or current is None
            or current[2] != label
        ):
            if current is not None:
                spans.append(tuple(current))
            current = [start, end, label]
        else:
            current[1] = end

    if current is not None:
        spans.append(tuple(current))

    return spans


def strict_prf(gold, pred):
    tp, fp, fn = Counter(), Counter(), Counter()

    for doc_id in set(gold) | set(pred):
        G = set(tuple(x) for x in gold.get(doc_id, []))
        P = set(tuple(x) for x in pred.get(doc_id, []))

        for entity in P:
            (tp if entity in G else fp)[entity[2]] += 1

        for entity in G - P:
            fn[entity[2]] += 1

    per_label = {}

    for label in LABELS:
        if tp[label] + fp[label] + fn[label] == 0:
            continue

        p = tp[label] / max(1, tp[label] + fp[label])
        r = tp[label] / max(1, tp[label] + fn[label])
        f = 2 * p * r / max(1e-12, p + r)

        per_label[label] = {
            "P": p, "R": r, "F1": f,
            "support": tp[label] + fn[label]
        }

    if not per_label:
        return {"P": 0.0, "R": 0.0, "F1": 0.0, "per_label": {}}

    return {
        "P": float(np.mean([x["P"] for x in per_label.values()])),
        "R": float(np.mean([x["R"] for x in per_label.values()])),
        "F1": float(np.mean([x["F1"] for x in per_label.values()])),
        "per_label": per_label
    }


def show_metrics(result, name=""):
    print(
        f"{name} | macro P={result['P']:.4f} "
        f"R={result['R']:.4f} F1={result['F1']:.4f}"
    )
    for label, values in result["per_label"].items():
        print(
            f"  {label:15s} P={values['P']:.3f} "
            f"R={values['R']:.3f} F1={values['F1']:.3f} "
            f"n={values['support']}"
        )

In [39]:

# ---------------------------- windows and batches ----------------------------

PAD = tokenizer.stoi["<pad>"]
BOS = tokenizer.stoi["<s>"]

def make_windows(n, win=WIN, step=STRIDE):
    if n <= win:
        return [(0, n)]
    starts = list(range(0, n - win + 1, step))
    if starts[-1] + win < n:
        starts.append(n - win)
    return [(s, min(s + win, n)) for s in starts]

def make_items(docs, step=STRIDE):
    items = []
    for doc_id, doc in docs.items():
        for a, b in make_windows(len(doc["ids"]), WIN, step):
            items.append({
                "doc_id": doc_id, "start": a, "end": b,
                "ids": [BOS] + doc["ids"][a:b],
                "y": [-100] + doc["y"][a:b]
            })
    return items

def collate(batch):
    max_len = max(len(item["ids"]) for item in batch)
    ids = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attn = torch.zeros((len(batch), max_len), dtype=torch.bool)
    for i, item in enumerate(batch):
        n = len(item["ids"])
        ids[i, :n] = torch.tensor(item["ids"], dtype=torch.long)
        labels[i, :n] = torch.tensor(item["y"], dtype=torch.long)
        attn[i, :n] = True
    return ids, labels, attn


In [40]:

# ---------------------------- hidden-state backbone / NER head ----------------------------

class Adapter(nn.Module):
    def __init__(self, d_model, rank):
        super().__init__()
        self.down = nn.Linear(d_model, rank)
        self.up = nn.Linear(rank, d_model)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.up(F.gelu(self.down(x)))

class NERBackbone(nn.Module):
    """
    Calls the actual Phase-1 TransformerBlock API.
    The language-model vocabulary head is bypassed; token hidden states are used.
    """
    def __init__(self, lm, causal=CAUSAL, adapter_dim=0):
        super().__init__()
        self.lm = lm
        self.causal = causal
        self.dim = lm.cfg.d_model
        self.adapters = (
            nn.ModuleList([Adapter(self.dim, adapter_dim) for _ in lm.blocks])
            if adapter_dim else None
        )

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.shape
        assert T <= self.lm.cfg.max_seq_len, (
            f"Window length {T} exceeds model max_seq_len={self.lm.cfg.max_seq_len}"
        )
        # Right padding. For bidirectional attention, bool True means allowed
        # keys in scaled_dot_product_attention.
        positions = torch.arange(T, device=input_ids.device)
        x = self.lm.tok_emb(input_ids)
        if self.lm.sin_emb is not None:
            x = x * math.sqrt(self.lm.cfg.d_model) + self.lm.sin_emb.pe[positions].to(x.dtype)
        x = self.lm.drop(x)

        if self.causal:
            attn_mask = None
        else:
            attn_mask = attention_mask[:, None, None, :].bool()

        aux_total = x.new_zeros(())
        for i, block in enumerate(self.lm.blocks):
            x, aux, _ = block(
                x,
                rope=self.lm.rope,
                positions=positions,
                kv_cache=None,
                attn_mask=attn_mask
            )
            aux_total = aux_total + aux
            if self.adapters is not None:
                x = self.adapters[i](x)

        return self.lm.final_norm(x), aux_total

class NERModel(nn.Module):
    def __init__(self, lm, head="linear", dropout=0.1, causal=CAUSAL, adapter_dim=0):
        super().__init__()
        self.backbone = NERBackbone(lm, causal=causal, adapter_dim=adapter_dim)
        d = self.backbone.dim
        self.drop = nn.Dropout(dropout)
        if head == "linear":
            self.head = nn.Linear(d, NL)
        elif head == "mlp":
            self.head = nn.Sequential(
                nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, NL)
            )
        else:
            raise ValueError("head must be 'linear' or 'mlp'")

    def forward(self, input_ids, attention_mask):
        hidden, aux_loss = self.backbone(input_ids, attention_mask)
        logits = self.head(self.drop(hidden))
        return logits, aux_loss


# ---------------------------- LoRA and LoRMA ----------------------------

class LoRALinear(nn.Module):
    """Additive low-rank update: base(x) + scale * B(A(x))."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.in_features) / math.sqrt(base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (x @ self.A.t() @ self.B.t()) * self.scale

class LoRMALinear(nn.Module):
    """Multiplicative output-side low-rank update: y + scale * B(A(y))."""
    def __init__(self, base, r=8, alpha=16):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.randn(r, base.out_features) / math.sqrt(base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        y = self.base(x)
        return y + (y @ self.A.t() @ self.B.t()) * self.scale


In [41]:

# Adapt attention projections only. Routed experts remain frozen and untouched.
LORA_TARGETS = {
    "q_proj", "k_proj", "v_proj", "o_proj",
    "q_down", "q_up", "kv_down", "kv_up", "k_rope_proj"
}

def inject_low_rank(lm, wrapper_cls, r=8, alpha=16):
    for block in lm.blocks:
        for parent_name, parent in list(block.named_modules()):
            if "experts" in parent_name.split(".") or "shared" in parent_name.split("."):
                continue
            for child_name, child in list(parent.named_children()):
                if child_name in LORA_TARGETS and isinstance(child, nn.Linear):
                    setattr(parent, child_name, wrapper_cls(child, r=r, alpha=alpha))


# ---------------------------- builders for methods ----------------------------

def build_model(variant, method, head="linear", r=LORA_R, k_top=TOP_K_BLOCKS):
    lm = load_phase1_model(variant)

    if method in ("lora", "lorma", "adapter", "partial", "kd", "student_noKD"):
        for p in lm.parameters():
            p.requires_grad = False

    adapter_dim = ADAPTER_DIM if method == "adapter" else 0

    if method == "partial":
        for block in lm.blocks[-k_top:]:
            for p in block.parameters():
                p.requires_grad = True
        for p in lm.final_norm.parameters():
            p.requires_grad = True

    elif method == "lora":
        inject_low_rank(lm, LoRALinear, r=r, alpha=2*r)
        for name, p in lm.named_parameters():
            if ".A" in name or ".B" in name:
                p.requires_grad = True

    elif method == "lorma":
        inject_low_rank(lm, LoRMALinear, r=r, alpha=2*r)
        for name, p in lm.named_parameters():
            if ".A" in name or ".B" in name:
                p.requires_grad = True

    model = NERModel(
        lm,
        head=head,
        dropout=0.1,
        causal=CAUSAL,
        adapter_dim=adapter_dim
    )

    # For adapter mode, the adapter parameters are new and trainable while LM is frozen.
    if method == "adapter":
        for p in model.backbone.lm.parameters():
            p.requires_grad = False
        for p in model.backbone.adapters.parameters():
            p.requires_grad = True

    # In all PEFT modes, train the NER classification head.
    for p in model.head.parameters():
        p.requires_grad = True

    return model.to(DEVICE)

def n_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ---------------------------- inference and evaluation ----------------------------

@torch.no_grad()
def predict_docs(model, docs, decode="viterbi", batch_size=EVAL_BATCH_SIZE):
    model.eval()
    predictions = {}
    for doc_id, doc in tqdm(docs.items(), desc="Predict"):
        n = len(doc["ids"])
        if n == 0:
            predictions[doc_id] = []
            continue
        acc = np.zeros((n, NL), dtype=np.float32)
        cnt = np.zeros(n, dtype=np.float32)
        windows = make_windows(n, WIN, WIN // 2)

        for start in range(0, len(windows), batch_size):
            chunk = windows[start:start + batch_size]
            batch = [
                {"ids": [BOS] + doc["ids"][a:b], "y": [-100] * (b-a+1)}
                for a, b in chunk
            ]
            ids, _, attn = collate(batch)
            ids, attn = ids.to(DEVICE), attn.to(DEVICE)
            with autocast_context():
                logits, _ = model(ids, attn)
            log_probs = logits.float().log_softmax(-1).cpu().numpy()
            for i, (a, b) in enumerate(chunk):
                acc[a:b] += log_probs[i, 1:b-a+1]
                cnt[a:b] += 1

        logp = acc / np.maximum(cnt[:, None], 1.0)
        tags = viterbi(logp) if decode == "viterbi" else logp.argmax(-1).tolist()
        predictions[doc_id] = tags_to_spans(tags, doc["offs"])
    return predictions

def evaluate(model, docs, decode="viterbi"):
    pred = predict_docs(model, docs, decode=decode)
    gold = {doc_id: doc["gold"] for doc_id, doc in docs.items()}
    return strict_prf(gold, pred), pred



In [42]:

# ---------------------------- training with resumable checkpoints ----------------------------

def class_weights(docs, power=0.5, cap=10.0):
    counts = np.bincount(
        np.concatenate([doc["y"] for doc in docs.values()]),
        minlength=NL
    ).astype(np.float64) + 1.0
    weights = np.clip((counts[0] / counts) ** power, 0, cap)
    weights[0] = 1.0
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

CW = class_weights(train_docs)

def checkpoint_paths(name):
    return (
        os.path.join(OUT, f"{name}_best.pt"),
        os.path.join(OUT, f"{name}_last.pt"),
        os.path.join(OUT, f"{name}_metadata.json"),
    )

def save_state(path, model, optimizer, epoch, best_f1, history, step=0, scaler=None):
    payload = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "scaler": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "step": step,
        "best_f1": best_f1,
        "history": history,
        "bio_labels": BIO,
        "labels": LABELS,
    }
    torch.save(payload, path)

def load_state(path, model, optimizer=None, scaler=None):
    state = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(state["model"], strict=True)
    if optimizer is not None and state.get("optimizer") is not None:
        optimizer.load_state_dict(state["optimizer"])
    if scaler is not None and state.get("scaler") is not None:
        scaler.load_state_dict(state["scaler"])
    return state

def train(
    model, name, epochs, lr, head_lr=HEAD_LR, batch_size=BATCH_SIZE,
    weight_decay=WEIGHT_DECAY, cw=CW, teacher=None, T=KD_T,
    alpha=KD_ALPHA, resume=True
):
    best_path, last_path, metadata_path = checkpoint_paths(name)
    items = make_items(train_docs)
    loader = DataLoader(
        items, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
        collate_fn=collate
    )

    backbone_params = [
        p for n, p in model.named_parameters()
        if p.requires_grad and not n.startswith("head.")
    ]
    head_params = [
        p for n, p in model.named_parameters()
        if p.requires_grad and n.startswith("head.")
    ]
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": lr})
    if head_params:
        groups.append({"params": head_params, "lr": head_lr})
    optimizer = torch.optim.AdamW(groups, weight_decay=weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    start_epoch, best_f1, history = 0, -1.0, []
    if resume and os.path.isfile(last_path):
        state = load_state(last_path, model, optimizer, scaler)
        start_epoch = int(state.get("epoch", 0))
        best_f1 = float(state.get("best_f1", -1.0))
        history = state.get("history", [])
        print(f"[{name}] resuming from epoch {start_epoch}")

    t0 = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(start_epoch, epochs):
        model.train()
        if teacher is not None:
            teacher.eval()
        random.shuffle(items)
        # Shuffle DataLoader's dataset list too by recreating it for this epoch.
        loader = DataLoader(
            items, batch_size=batch_size, shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=(DEVICE.type == "cuda"),
            persistent_workers=(NUM_WORKERS > 0),
            collate_fn=collate
        )
        total_loss, batches = 0.0, 0

        for ids, y, attn in tqdm(loader, desc=f"{name} epoch {epoch+1}/{epochs}"):
            ids, y, attn = ids.to(DEVICE), y.to(DEVICE), attn.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP
            ):
                logits, aux = model(ids, attn)
                ce = F.cross_entropy(
                    logits.float().reshape(-1, NL),
                    y.reshape(-1),
                    weight=cw,
                    ignore_index=-100
                )
                loss = ce + model.backbone.lm.cfg.aux_loss_coef * aux

                if teacher is not None:
                    with torch.no_grad():
                        t_logits, _ = teacher(ids, attn)
                    selected = y != -100
                    if selected.any():
                        student_logp = F.log_softmax(logits.float()[selected] / T, dim=-1)
                        teacher_prob = F.softmax(t_logits.float()[selected] / T, dim=-1)
                        kd = F.kl_div(
                            student_logp, teacher_prob, reduction="batchmean"
                        ) * (T * T)
                        loss = alpha * loss + (1 - alpha) * kd

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0
            )
            scaler.step(optimizer)
            scaler.update()

            total_loss += float(loss.detach().item())
            batches += 1

        val_result, _ = evaluate(model, val_docs)
        avg_loss = total_loss / max(1, batches)
        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "P": val_result["P"],
            "R": val_result["R"],
            "F1": val_result["F1"],
        })
        print(
            f"[{name}] epoch {epoch+1}/{epochs} loss={avg_loss:.4f} "
            f"| val P={val_result['P']:.4f} R={val_result['R']:.4f} "
            f"F1={val_result['F1']:.4f}"
        )

        if val_result["F1"] > best_f1:
            best_f1 = val_result["F1"]
            save_state(best_path, model, None, epoch + 1, best_f1, history)
            print("Saved best:", best_path)

        save_state(last_path, model, optimizer, epoch + 1, best_f1, history, scaler=scaler)

    # Restore best weights before returning.
    best_state = load_state(best_path, model)
    peak_gb = (
        torch.cuda.max_memory_allocated() / (1024**3)
        if DEVICE.type == "cuda" else float("nan")
    )
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump({
            "name": name,
            "epochs_requested": epochs,
            "best_epoch": best_state.get("epoch"),
            "best_val_f1": best_f1,
            "bio_labels": BIO,
            "labels": LABELS,
            "causal": CAUSAL,
            "window": WIN,
            "stride": STRIDE,
        }, f, indent=2)
    return {
        "best_val_f1": best_f1,
        "train_sec": time.time() - t0,
        "peak_gb": peak_gb,
        "hist": history,
        "best_path": best_path,
        "last_path": last_path,
    }



In [43]:

# ---------------------------- test data and submission writer ----------------------------

def load_test_docs():
    with open(dl("Test/fold3-text.json"), encoding="utf-8") as f:
        texts = json.load(f)
    texts = {str(k): v for k, v in texts.items()}
    docs = {}
    for doc_id, text in tqdm(texts.items(), desc="Encode test"):
        ids, offs = encode_with_offsets(text)
        docs[doc_id] = {
            "text": text, "ids": ids, "offs": offs,
            "y": [0] * len(ids), "gold": []
        }
    return texts, docs

def write_test(model, name):
    test_texts, test_docs = load_test_docs()
    pred = predict_docs(model, test_docs)
    js = {
        doc_id: [
            [s, e, label, test_texts[doc_id][s:e]]
            for s, e, label in spans
        ]
        for doc_id, spans in pred.items()
    }
    json_path = os.path.join(OUT, f"prediction_{name}.json")
    zip_path = os.path.join(OUT, f"submission_{name}.zip")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(js, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")
    print("Wrote:", json_path)
    print("Wrote:", zip_path)
    return json_path, zip_path



In [77]:


# ---------------------------- one complete experiment ----------------------------

RESULTS, MODELS = {}, {}

def run_experiment(variant, method, epochs=None, lr=None, head="linear", resume=True):
    """
    method: full, partial, lora, lorma, adapter, kd, student_noKD
    Unique checkpoint names are created per variant and method.
    """
    name = f"{variant}__{method}"
    epochs = epochs or EPOCHS[method]
    lr = lr or LR[method]

    # Avoid silently retraining a completed run. Load best if already present.
    best_path, last_path, _ = checkpoint_paths(name)
    if os.path.isfile(best_path) and not os.path.isfile(last_path):
        print(f"[{name}] best checkpoint exists; loading without training.")
        model = build_model(variant, method, head=head)
        state = load_state(best_path, model)
        result, pred = evaluate(model, val_docs)
        info = {
            "best_val_f1": state.get("best_f1", result["F1"]),
            "train_sec": 0.0,
            "peak_gb": float("nan"),
            "hist": state.get("history", []),
        }
    else:
        model = build_model(variant, method, head=head)
        tot, tr = n_params(model)
        print(
            f"\n=== {name}: total {tot/1e6:.2f}M, trainable {tr/1e6:.2f}M "
            f"({100*tr/max(1,tot):.2f}%) ==="
        )
        info = train(
            model, name, epochs=epochs, lr=lr, resume=resume,
            teacher=None
        )
        result, pred = evaluate(model, val_docs)

    tot, tr = n_params(model)
    info.update({
        "variant": variant,
        "method": method,
        "total_M": tot / 1e6,
        "trainable_M": tr / 1e6,
        "P": result["P"],
        "R": result["R"],
        "F1": result["F1"],
        "per_label": result["per_label"],
        "errors": error_analysis(gold_v, pred),
    })
    RESULTS[name] = info
    MODELS[name] = model
    show(result, f"[{name}] FINAL VALIDATION")
    write_test(model, name)
    return model


In [45]:

# ---------------------------- knowledge distillation helpers ----------------------------

def build_kd_student(variant, keep=KD_KEEP_BLOCKS):
    """
    Build a 4-block student from selected pretrained Phase-1 blocks.
    Blocks are deep-copied from the original model so their original MoE/dense
    structure is preserved exactly (important for layer-specific FFN shapes).
    """
    teacher_lm = load_phase1_model(variant)
    if max(keep) >= len(teacher_lm.blocks):
        raise ValueError(f"KD_KEEP_BLOCKS {keep} exceeds model depth {len(teacher_lm.blocks)}")

    student_lm = copy.deepcopy(teacher_lm)
    student_lm.blocks = nn.ModuleList([
        copy.deepcopy(teacher_lm.blocks[i]) for i in keep
    ])
    # The forward method uses cfg.max_seq_len and cfg.aux_loss_coef; n_layers is
    # only construction metadata after the blocks have been copied.
    student_lm.cfg = copy.deepcopy(teacher_lm.cfg)
    student_lm.cfg.n_layers = len(keep)
    return student_lm.to(DEVICE)

def run_kd_experiment(variant, teacher_model, epochs=None, resume=True):
    method = "kd"
    name = f"{variant}__{method}"
    epochs = epochs or EPOCHS["kd_student"]
    best_path, last_path, _ = checkpoint_paths(name)

    student_lm = build_kd_student(variant)
    student = NERModel(student_lm, head="linear", dropout=0.1, causal=CAUSAL).to(DEVICE)
    # KD student is fully fine-tuned, as in the original Phase-2 experiment.
    for p in student.parameters():
        p.requires_grad = True

    if os.path.isfile(best_path) and not os.path.isfile(last_path):
        state = load_state(best_path, student)
        result, pred = evaluate(student, val_docs)
        info = {"best_val_f1": state.get("best_f1", result["F1"]), "train_sec": 0.0,
                "peak_gb": float("nan"), "hist": state.get("history", [])}
    else:
        tot, tr = n_params(student)
        print(f"\n=== {name}: total {tot/1e6:.2f}M, trainable {tr/1e6:.2f}M ===")
        info = train(
            student, name, epochs=epochs, lr=LR["kd_student"],
            resume=resume, teacher=teacher_model, T=KD_T, alpha=KD_ALPHA
        )
        result, pred = evaluate(student, val_docs)

    tot, tr = n_params(student)
    info.update({
        "variant": variant, "method": method,
        "total_M": tot / 1e6, "trainable_M": tr / 1e6,
        "P": result["P"], "R": result["R"], "F1": result["F1"],
        "per_label": result["per_label"], "errors": error_analysis(gold_v, pred),
    })
    RESULTS[name] = info
    MODELS[name] = student
    show(result, f"[{name}] FINAL VALIDATION")
    write_test(student, name)
    return student

def run_no_kd_control(variant, epochs=None, resume=True):
    method = "student_noKD"
    name = f"{variant}__{method}"
    epochs = epochs or EPOCHS[method]
    best_path, last_path, _ = checkpoint_paths(name)

    student_lm = build_kd_student(variant)
    student = NERModel(student_lm, head="linear", dropout=0.1, causal=CAUSAL).to(DEVICE)
    for p in student.parameters():
        p.requires_grad = True

    if os.path.isfile(best_path) and not os.path.isfile(last_path):
        state = load_state(best_path, student)
        result, pred = evaluate(student, val_docs)
        info = {"best_val_f1": state.get("best_f1", result["F1"]), "train_sec": 0.0,
                "peak_gb": float("nan"), "hist": state.get("history", [])}
    else:
        info = train(
            student, name, epochs=epochs, lr=LR["student_noKD"],
            resume=resume, teacher=None
        )
        result, pred = evaluate(student, val_docs)

    tot, tr = n_params(student)
    info.update({
        "variant": variant, "method": method,
        "total_M": tot / 1e6, "trainable_M": tr / 1e6,
        "P": result["P"], "R": result["R"], "F1": result["F1"],
        "per_label": result["per_label"], "errors": error_analysis(gold_v, pred),
    })
    RESULTS[name] = info
    MODELS[name] = student
    show(result, f"[{name}] FINAL VALIDATION")
    write_test(student, name)
    return student



In [67]:

# ---------------------------- full suite for ONE variant ----------------------------

def run_variant_suite(variant, run_kd=True):
    """
    Runs all original Phase-2 methods for this Phase-1 architecture:
      full FT, partial top-3, LoRA, LoRMA, adapters, KD, no-KD control.
    Checkpoints are kept per experiment so completed runs are not repeated.
    """
    print("\n" + "=" * 80)
    print("PHASE-2 VARIANT:", variant)
    print("=" * 80)

    # Exp 1: full fine-tuning; this is the KD teacher.
    teacher = run_experiment(variant, "full", epochs=EPOCHS["full_ft"], lr=LR["full_ft"])

    # Exp 2: partial fine-tuning, top three blocks.
    # run_experiment(variant, "partial", epochs=EPOCHS["partial_top3"], lr=LR["partial_top3"])

    # Exp 3: LoRA rank 8.
    # run_experiment(variant, "lora", epochs=EPOCHS["lora_r8"], lr=LR["lora_r8"])

    # Exp 4: LoRMA rank 8.
    # run_experiment(variant, "lorma", epochs=EPOCHS["lorma_r8"], lr=LR["lorma_r8"])

    # Exp 5: Houlsby-style adapters.
    # run_experiment(variant, "adapter", epochs=EPOCHS["adapter64"], lr=LR["adapter64"])

    # Exp 6: KD student + Exp 7: same student without KD.
    # if run_kd:
        # run_kd_experiment(variant, teacher, epochs=EPOCHS["kd_student"])
        # run_no_kd_control(variant, epochs=EPOCHS["student_noKD"])

    return RESULTS


# ---------------------------- saved-model loader / test-only inference ----------------------------

def load_trained_ner(variant, method, head="linear"):
    """
    Load a Phase-2 best checkpoint without training.
    method names: full, partial, lora, lorma, adapter, kd, student_noKD.
    """
    name = f"{variant}__{method}"
    best_path, _, _ = checkpoint_paths(name)
    if not os.path.isfile(best_path):
        raise FileNotFoundError(f"No saved best Phase-2 checkpoint: {best_path}")

    if method in ("kd", "student_noKD"):
        lm = build_kd_student(variant)
        model = NERModel(lm, head=head, dropout=0.1, causal=CAUSAL).to(DEVICE)
        for p in model.parameters():
            p.requires_grad = True
    else:
        model = build_model(variant, method, head=head)

    state = load_state(best_path, model)
    model.eval()
    print(f"Loaded saved NER model: {name} | best F1={state.get('best_f1')}")
    return model

def decoding_ablation(model, docs=val_docs):
    for dec in ("greedy", "viterbi"):
        result, _ = evaluate(model, docs, decode=dec)
        print(f"decode={dec}: macro F1={result['F1']:.4f}")

def summarize_results():
    rows = []
    for name, v in RESULTS.items():
        rows.append({
            "experiment": name,
            "total_M": round(v.get("total_M", float("nan")), 2),
            "trainable_M": round(v.get("trainable_M", float("nan")), 2),
            "P": round(v.get("P", float("nan")), 4),
            "R": round(v.get("R", float("nan")), 4),
            "F1": round(v.get("F1", float("nan")), 4),
            "train_min": round(v.get("train_sec", 0.0) / 60, 2),
            "peak_GB": round(v.get("peak_gb", float("nan")), 2),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("F1", ascending=False)
        print(df.to_string(index=False))
        df.to_csv(os.path.join(OUT, "phase2_all_results.csv"), index=False)
        best_name = df.iloc[0]["experiment"]
        print("\nBest validation experiment:", best_name)
        print("Error breakdown:", RESULTS[best_name].get("errors", {}))
        if best_name in MODELS:
            decoding_ablation(MODELS[best_name])
    return df


In [54]:
# First test one variant by running:
run_variant_suite("mla_rope_moe")
#
# When satisfied, run all four (this is computationally expensive):
  




PHASE-2 VARIANT: mla_rope_moe
Loaded Phase-1 mla_rope_moe from /kaggle/working/phase1-models/best_models/abl_mla_rope_moe_best.pt

=== mla_rope_moe__full: total 154.32M, trainable 154.32M (100.00%) ===


mla_rope_moe__full epoch 1/10:   0%|          | 0/185 [00:00<?, ?it/s]

Predict:   0%|          | 0/14 [00:00<?, ?it/s]

[mla_rope_moe__full] epoch 1/10 loss=0.6295 | val P=0.2872 R=0.4842 F1=0.3407
Saved best: /kaggle/working/lner_runs/mla_rope_moe__full_best.pt


mla_rope_moe__full epoch 2/10:   0%|          | 0/185 [00:00<?, ?it/s]

Predict:   0%|          | 0/14 [00:00<?, ?it/s]

... (72 outputs left out)

Encode test:   0%|          | 0/35 [00:00<?, ?it/s]

Predict:   0%|          | 0/35 [00:00<?, ?it/s]

Wrote: /kaggle/working/lner_runs/prediction_mla_rope_moe__kd.json
Wrote: /kaggle/working/lner_runs/submission_mla_rope_moe__kd.zip


{'mla_rope_moe__lora': {'best_val_f1': 0.4796135750422435,
  'train_sec': 0.42087769508361816,
  'peak_gb': 5.409163475036621,
  'hist': [{'epoch': 1,
    'train_loss': 0.7280646531968503,
    'P': 0.28517771267144393,
    'R': 0.45367264679273656,
    'F1': 0.33833605464222344},
   {'epoch': 2,
    'train_loss': 0.35338305734299325,
    'P': 0.34086132104146233,
    'R': 0.5021628757086428,
    'F1': 0.3955280218012656},
   {'epoch': 3,
    'train_loss': 0.2863300751995396,
    'P': 0.37798339794773295,
    'R': 0.48526883276127614,
    'F1': 0.41514803115916504},
   {'epoch': 4,
    'train_loss': 0.2482110278831946,
    'P': 0.42774505463264895,
    'R': 0.5080049238200917,
    'F1': 0.45390021775887707},
   {'epoch': 5,
    'train_loss': 0.22896929834340068,
    'P': 0.4252857302106318,
    'R': 0.5157446965571724,
    'F1': 0.451379405942584},
   {'epoch': 6,
    'train_loss': 0.21335768611044498,
    'P': 0.4166660026323285,
    'R': 0.5411352692692495,
    'F1': 0.463994978257575

In [55]:
name = "mla_rope_moe__lora"

if name in RESULTS:
    show(RESULTS[name], f"[{name}] FINAL VALIDATION")
else:
    print("Result not found; inspect the saved checkpoints before rerunning.")

[mla_rope_moe__lora] FINAL VALIDATION | macro P 0.4489 R 0.5425 F1 0.4796
  APP             P 0.080 R 0.213 F1 0.117 n=61
  RESP            P 0.222 R 0.212 F1 0.217 n=66
  A.COUNSEL       P 0.500 R 0.385 F1 0.435 n=26
  R.COUNSEL       P 0.484 R 0.714 F1 0.577 n=21
  JUDGE           P 0.561 R 0.719 F1 0.630 n=64
  WIT             P 0.444 R 0.323 F1 0.374 n=62
  AUTH            P 0.257 R 0.668 F1 0.371 n=190
  COURT           P 0.753 R 0.809 F1 0.780 n=136
  STAT            P 0.590 R 0.774 F1 0.669 n=212
  PREC            P 0.210 R 0.264 F1 0.234 n=201
  DATE            P 0.900 R 0.946 F1 0.923 n=315
  CASENO          P 0.384 R 0.484 F1 0.429 n=161


In [61]:
for variant in VARIANTS:
    run_variant_suite(variant)
df_results = summarize_results()


PHASE-2 VARIANT: gqa_rope_moe
Loaded Phase-1 gqa_rope_moe from /kaggle/working/phase1-models/best_models/abl_gqa_rope_moe_best.pt

=== gqa_rope_moe__lora: total 152.78M, trainable 0.66M (0.43%) ===
[gqa_rope_moe__lora] resuming from epoch 10


Predict:   0%|          | 0/14 [00:00<?, ?it/s]

[gqa_rope_moe__lora] FINAL VALIDATION | macro P 0.4586 R 0.5787 F1 0.5082
  APP             P 0.165 R 0.213 F1 0.186 n=61
  RESP            P 0.233 R 0.318 F1 0.269 n=66
  A.COUNSEL       P 0.560 R 0.538 F1 0.549 n=26
  R.COUNSEL       P 0.533 R 0.762 F1 0.627 n=21
  JUDGE           P 0.512 R 0.656 F1 0.575 n=64
  WIT             P 0.316 R 0.484 F1 0.382 n=62
  AUTH            P 0.368 R 0.574 F1 0.449 n=190
  COURT           P 0.714 R 0.846 F1 0.774 n=136
  STAT            P 0.640 R 0.722 F1 0.678 n=212
  PREC            P 0.246 R 0.308 F1 0.274 n=201
  DATE            P 0.878 R 0.933 F1 0.905 n=315
  CASENO          P 0.338 R 0.590 F1 0.430 n=161


Encode test:   0%|          | 0/35 [00:00<?, ?it/s]

Predict:   0%|          | 0/35 [00:00<?, ?it/s]

Wrote: /kaggle/working/lner_runs/prediction_gqa_rope_moe__lora.json
Wrote: /kaggle/working/lner_runs/submission_gqa_rope_moe__lora.zip

PHASE-2 VARIANT: mla_rope_moe
Loaded Phase-1 mla_rope_moe from /kaggle/working/phase1-models/best_models/abl_mla_rope_moe_best.pt

=== mla_rope_moe__lora: total 155.24M, trainable 0.94M (0.60%) ===
[mla_rope_moe__lora] resuming from epoch 10


... (30 outputs left out)

Predict:   0%|          | 0/14 [00:00<?, ?it/s]

decode=greedy: macro F1=0.4196


Predict:   0%|          | 0/14 [00:00<?, ?it/s]

decode=viterbi: macro F1=0.5421


In [78]:
run_variant_suite("gqa_rope_moe")


PHASE-2 VARIANT: gqa_rope_moe
Loaded Phase-1 gqa_rope_moe from /kaggle/working/phase1-models/best_models/abl_gqa_rope_moe_best.pt

=== gqa_rope_moe__full: total 152.15M, trainable 152.15M (100.00%) ===
[gqa_rope_moe__full] resuming from epoch 13


Predict:   0%|          | 0/14 [00:00<?, ?it/s]

[gqa_rope_moe__full] FINAL VALIDATION | macro P 0.5148 R 0.5770 F1 0.5421
  APP             P 0.184 R 0.262 F1 0.216 n=61
  RESP            P 0.375 R 0.364 F1 0.369 n=66
  A.COUNSEL       P 0.538 R 0.538 F1 0.538 n=26
  R.COUNSEL       P 0.560 R 0.667 F1 0.609 n=21
  JUDGE           P 0.598 R 0.812 F1 0.689 n=64
  WIT             P 0.371 R 0.371 F1 0.371 n=62
  AUTH            P 0.278 R 0.311 F1 0.294 n=190
  COURT           P 0.831 R 0.831 F1 0.831 n=136
  STAT            P 0.682 R 0.840 F1 0.753 n=212
  PREC            P 0.279 R 0.358 F1 0.314 n=201
  DATE            P 0.949 R 0.949 F1 0.949 n=315
  CASENO          P 0.532 R 0.621 F1 0.573 n=161


Encode test:   0%|          | 0/35 [00:00<?, ?it/s]

Predict:   0%|          | 0/35 [00:00<?, ?it/s]

Wrote: /kaggle/working/lner_runs/prediction_gqa_rope_moe__full.json
Wrote: /kaggle/working/lner_runs/submission_gqa_rope_moe__full.zip


{'gqa_rope_moe__full': {'best_val_f1': 0.5421140649361802,
  'train_sec': 0.36159515380859375,
  'peak_gb': 4.139093399047852,
  'hist': [{'epoch': 1,
    'train_loss': 0.4950189606563465,
    'P': 0.3628913263649533,
    'R': 0.5245124290653105,
    'F1': 0.4132105189054612},
   {'epoch': 2,
    'train_loss': 0.22254948358278018,
    'P': 0.43082242241126156,
    'R': 0.5354653187844991,
    'F1': 0.4667306050062785},
   {'epoch': 3,
    'train_loss': 0.17590325756652936,
    'P': 0.456558305850493,
    'R': 0.5876455272570023,
    'F1': 0.5068850560255279},
   {'epoch': 4,
    'train_loss': 0.1562358725715328,
    'P': 0.5146588360585023,
    'R': 0.5283560386544748,
    'F1': 0.5094250833132108},
   {'epoch': 5,
    'train_loss': 0.1479484432452434,
    'P': 0.46281700197904313,
    'R': 0.5866973007951958,
    'F1': 0.5073342594717146},
   {'epoch': 6,
    'train_loss': 0.1445888447600442,
    'P': 0.507986491845394,
    'R': 0.5518256188008479,
    'F1': 0.5242640347046233},
   {'